# Using decoding compression on the AIME25 Math Dataset

This notebook demonstrates how to compress during text generation.
We use `nvidia/OpenMath-Nemotron-7B` to solve math problems from the AIME25 dataset. For each problem, the model generates an answer in a boxed format (e.g., `\boxed{42}`).

To optimize memory usage during long-context generation, the notebook applies key-value cache compression during decoding.
Compression periodically reduces the cache size by keeping only the most relevant tokens, enabling efficient inference without sacrificing answer quality.

In [ ]:
from transformers import pipeline
from datasets import load_dataset

from kvpress import (
    DecodingPress,
    ExpectedAttentionPress,
    KnormPress,
    ObservedAttentionPress,
    RandomPress,
    SnapKVPress,
    StreamingLLMPress,
    TOVAPress,
)

In [ ]:
# Load a sample
dataset = load_dataset("math-ai/aime25")
sample = dataset["test"][0]
sample

In [ ]:
# Load the pipeline
device = "cuda:0"
ckpt = "nvidia/OpenMath-Nemotron-7B"
attn_implementation = "flash_attention_2"
pipe = pipeline("kv-press-text-generation", model=ckpt, device=device, dtype="auto", model_kwargs={"attn_implementation":attn_implementation})

In [ ]:
# Run the pipeline without compression

question = sample["problem"]
true_answer = sample["answer"]
pred_answer = pipe(" ", question=question, press=None, max_new_tokens=16_000)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")

In [ ]:
# Run the pipeline with compression

compression_interval = 1024  # compress every compression_steps
target_size = 512 # number of tokens to keep after compression. Note that actual cache size lies in [target_size, compression_interval]

press = DecodingPress(base_press=ExpectedAttentionPress(), compression_interval=compression_interval, target_size=target_size)

question = sample["problem"]
true_answer = sample["answer"]
pred_answer = pipe(" ", question=question, press=press, max_new_tokens=16_000)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")